# Declare a new run

Bare-minimum template for starting a new run: define its sources, its
tokenizer (reused if one already matches), a dataset and a pretraining config
under a fresh `run_id`, then check and declare it against the volume through
`lab` -- same shape as `notebooks/declare.ipynb`, but through the `lab` API
(environment/target-aware, works the same whether this runs in JupyterLab or
locally) instead of calling `Artifact.declare()` directly, and living at the
repo root so `import lab` needs no path setup.

Copy this notebook per run and change `RUN_ID` and the knobs below. This
only declares up through the `Pretraining` artifact -- it doesn't run any
jobs. `lab.declare(..., commit=True, cell=DEFINITION)` is also what creates
`runs/{run_id}/notebook.ipynb` on the volume the first time: `DEFINITION` --
this cell's own source -- is what gets seeded into it, so that notebook
picks up exactly where this one left off.

In [1]:
import lab

lab.init(target="modal")  # this run is declared straight to the real volume

In [2]:
from dag.artifact import Resources
from mappeddatasets.artifact import MappedDataSet
from models.mock.artifact import ModelParameters, Pretraining, PretrainingConfig
from sources.artifact import Source
from tokenizers.bpe import Tokenizer as BPETokenizer

odyssey = Source(name="odyssey", url="https://www.gutenberg.org/cache/epub/1727/pg1727.txt")
mobydick = Source(name="mobydick", url="https://www.gutenberg.org/cache/epub/2701/pg2701.txt")
romeojuliet = Source(name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt")
montecristo = Source(name="montecristo", url="https://www.gutenberg.org/cache/epub/1184/pg1184.txt")
pride = Source(name="pride", url="https://www.gutenberg.org/cache/epub/1342/pg1342.txt")
frankenstein = Source(name="frankenstein", url="https://www.gutenberg.org/cache/epub/84/pg84.txt")
greatexpectations = Source(name="greatexpectations", url="https://www.gutenberg.org/cache/epub/1400/pg1400.txt")
dracula = Source(name="dracula", url="https://www.gutenberg.org/cache/epub/345/pg345.txt")

RUN_ID = "RUN_EXAMPLE"  # <- change this per run

tokenizer = BPETokenizer(
    vocab_size=350,
    special_tokens=("<pad>", "<unk>"),
    sources=(odyssey, mobydick, dracula),
)

mappedset = MappedDataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=[montecristo, mobydick, frankenstein],
    valid_sources=[dracula, pride],
)

model_parameters = ModelParameters(hidden_size=64, num_layers=2)
config = PretrainingConfig(
    total_steps=6000, batch_size=32, lr=1e-3, seed=1, checkpoint_every=500
)

pretraining = Pretraining(
    run_id=RUN_ID,
    dataset=mappedset,
    tokenizer=tokenizer,
    model_parameters=model_parameters,
    config=config,
    allocated_resources=Resources(gpu_type="T4", gpu_count=1),
)

DEFINITION = In[-1]  # this cell's own source, for the notebook declare() drops on the volume

## Declare

`lab.declare(pretraining)` resolves the request and reconciles it against
`lab.target` without writing anything -- everything shared (sources, and the
tokenizer if it matches one already declared) should read `done`;
everything new to this run should read `new`. `commit=True` declares
whatever's `new`, refusing outright if anything's inconsistent. `run_id` is
read straight off `pretraining` -- no need to pass it separately.

In [3]:
lab.declare(pretraining)  # preview, no writes

run RUN_EXAMPLE under /storage
  new        sources/odyssey
  new        sources/mobydick
  new        sources/dracula
  new        tokenizers/bpe-350-47bbb01e5a
  new        sources/montecristo
  new        tokenizers/bpe-350-47bbb01e5a/bin/montecristo
  new        tokenizers/bpe-350-47bbb01e5a/bin/mobydick
  new        sources/frankenstein
  new        tokenizers/bpe-350-47bbb01e5a/bin/frankenstein
  new        tokenizers/bpe-350-47bbb01e5a/bin/dracula
  new        sources/pride
  new        tokenizers/bpe-350-47bbb01e5a/bin/pride
  new        mappeddatasets/mapped-927326d136
  new        runs/RUN_EXAMPLE/pretraining

14 new
ok -- 14 to declare


In [4]:
lab.declare(pretraining, commit=True, cell=DEFINITION)  # commits to the volume, seeds runs/RUN_EXAMPLE/notebook.ipynb

run RUN_EXAMPLE under /storage
  declared   sources/odyssey
  declared   sources/mobydick
  declared   sources/dracula
  declared   tokenizers/bpe-350-47bbb01e5a
  declared   sources/montecristo
  declared   tokenizers/bpe-350-47bbb01e5a/bin/montecristo
  declared   tokenizers/bpe-350-47bbb01e5a/bin/mobydick
  declared   sources/frankenstein
  declared   tokenizers/bpe-350-47bbb01e5a/bin/frankenstein
  declared   tokenizers/bpe-350-47bbb01e5a/bin/dracula
  declared   sources/pride
  declared   tokenizers/bpe-350-47bbb01e5a/bin/pride
  declared   mappeddatasets/mapped-927326d136
  declared   runs/RUN_EXAMPLE/pretraining

14 declared
ok -- 0 to declare
